<a href="https://colab.research.google.com/github/shagunsri09/Digital-product-funnel-and-retention-analytics-/blob/main/Digital_product_funnel_and_retention_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#admission no:- 2026b15310466
#name: Shagun Srivastava
#serial no: 39
#Github :- https://github.com/shagunsri09/Digital-product-funnel-and-retention-analytics-.git
from google.colab import files

In [ ]:
upload = files.upload()
filename = next(iter(upload))

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
df = pd.read_csv(filename)

In [ ]:
df.head()

In [ ]:
#Event grain represents what one row in dataset represents.
#In this project each row is representing one individual user activity at specific timestamps.

#check deduplicate events.
df.duplicated(subset="event_id").sum()

In [ ]:
df_clean = df.drop_duplicates(subset = "event_id")

In [ ]:
df_clean["event_id"].duplicated().sum()

In [ ]:
df_clean.head()

In [ ]:
#convert timestamp into datetime format
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])

In [ ]:
#sort events by user, session and timestamp
df_sorted = df_clean.sort_values(by=["user_id","session_id","timestamp"])

In [ ]:
df_sorted.head()

In [ ]:
#check user session mapping.
df_sorted.groupby("session_id")["user_id"].nunique().value_counts()

In [ ]:
#check first and last event of each session
df_sorted.groupby("session_id")["event_name"].agg(["first","last"]).value_counts()

In [ ]:
#define expected event order
event_order = {
    "app_open": 1,
    "product_view": 2,
    "add_to_cart": 3,
    "checkout": 4,
    "purchase": 5
}

In [ ]:
#assign order number
df_sorted["event_order"] = df_sorted["event_name"].map(event_order)

In [ ]:
from types import LambdaType
#validate event ordering within each session
df_sorted.groupby("session_id")["event_order"].apply(lambda x: x.is_monotonic_increasing).value_counts()

In [ ]:
# count unique users at each funnel  stage
funnel_counts = df_clean.groupby("event_name")["user_id"].nunique()

In [ ]:
#arrange stages in funnel order
funnel_counts = funnel_counts.reindex([
    "app_open",
    "product_view",
    "add_to_care",
    "checkout",
    "purchase"
])
funnel_counts

In [ ]:
#calculate conversion rates at each funnel stages
conversion_rate = (funnel_counts / funnel_counts.iloc[0]) * 100
conversion_rate

In [ ]:
user_purchase = df_clean.groupby("user_id")["event_name"].apply(lambda x: int("purchase" in x.values))

In [ ]:
#bootstrap confidence interval
bootstrap_rates = []
for i in range(1000):
  sample = np.random.choice(user_purchase.values, size=len(user_purchase),replace=True)
  bootstrap_rates.append(np.mean(sample) * 100)

  np.percentile(bootstrap_rates, [2.5, 97.5])

In [ ]:
np.percentile(bootstrap_rates , [2.5, 97.5])

In [ ]:
device_purchase = df_clean.groupby("device")["event_name"].apply(lambda x: (x=="purchase").sum())

In [ ]:
device_purchase

In [ ]:
#create cohort and activity months
df_cohort = df_clean.copy()

In [ ]:
df_cohort["activity_month"] = df_cohort["timestamp"].dt.to_period("M")

In [ ]:
df_cohort["cohort_month"] = df_cohort.groupby("user_id")["timestamp"].transform("min").dt.to_period("M")

In [ ]:
df_cohort.head()

In [ ]:
#count unique users in each cohort and activity month
cohort_data = df_cohort.groupby(["cohort_month","activity_month"])["user_id"].nunique().reset_index()

In [ ]:
cohort_data

In [ ]:
cohort_data["cohort_index"] = (
    (pd.to_datetime(cohort_data["activity_month"]).dt.year -
     pd.to_datetime(cohort_data["cohort_month"]).dt.year) * 12 +
    (pd.to_datetime(cohort_data["activity_month"]).dt.month -
     pd.to_datetime(cohort_data["cohort_month"]).dt.month)
)

In [ ]:
cohort_data.head()

In [ ]:
cohort_pivot = cohort_data.pivot(index="cohort_month",
                                 columns="cohort_index",
                                 values="user_id")
cohort_pivot

In [ ]:
#calculate cohort retention percentage
cohort_size = cohort_pivot.iloc[:,0]
retention= cohort_pivot.divide(cohort_size, axis=0) * 100

In [ ]:
retention

In [ ]:
user_sessions = df_clean.groupby("user_id")["session_id"].nunique()

In [ ]:
user_sessions

In [ ]:
repeat_users = (user_sessions>1).sum()

In [ ]:
repeat_users

In [ ]:
repeat_engagement_rate = (repeat_users / user_sessions.count()) * 100

In [ ]:
repeat_engagement_rate

In [ ]:
device_funnel = df_clean.groupby(["device","event_name"])["user_id"].nunique().unstack(fill_value=0)
device_funnel = device_funnel.reindex(columns=["app_open","product_view","add_to_cart","checkout",
                                               "purchase"],
                                      fill_value=0)


In [ ]:
device_funnel

In [ ]:
source_funnel = df_clean.groupby(["acquisition_source","event_name"])["user_id"].nunique()

In [ ]:
source_funnel = source_funnel.unstack(fill_value=0)

In [ ]:
source_funnel

In [ ]:
cohort_funnel = df_cohort.groupby(["cohort_month","event_name"])["user_id"].nunique().unstack(fill_value=0)

cohort_funnel

In [ ]:
df_sorted["time_diff"] = df_sorted.groupby("user_id")["timestamp"].diff()

In [ ]:
df_sorted[["user_id","timestamp","time_diff"]].head()


In [ ]:
df_sorted["new_session_30"] = df_sorted["time_diff"].isna()
(df_sorted["time_diff"]> pd.Timedelta(minutes=30))

In [ ]:
df_sorted["new_session_30"].value_counts()

In [ ]:
df_sorted["new_session_60"] = df_sorted["time_diff"].isna()
(df_sorted["time_diff"]> pd.Timedelta(minutes=60))

In [ ]:
df_sorted["new_session_60"].value_counts()

In [ ]:
df_sorted[["new_session_30","new_session_60"]].sum()

In [ ]:
#Funnel 1: App Open -> product view -> purchase
#Funnel 2: App open -> add to cart -> purchase

In [ ]:
#observational findings: the number of users decreases at later funnel stages.




 #recommended experiment: text weather simplifying checkout improves purchase conversions.

#causal result: a control experiment is needed to confirm causality.